# ArborGrid · 05 Contrato Ouro → MVP (Requisito 08)

**Tabela/visão consumida pelo MVP:** `workspace.gs.gold_fato_risco_arboreo` (nome definido no contrato da API ArborGrid).
Uma linha por árvore com o **risco do dia de referência mais recente** (as 48h seguintes).

| Coluna | Tipo | Significado |
|---|---|---|
| `geosampa_id` | STRING | árvore no cadastro da Prefeitura (chave do MERGE no MVP) |
| `distrito` | STRING | distrito oficial |
| `composite_risk_score` | DOUBLE (0–100) | score composto: risco da árvore × fator vento |
| `priority_tier` | STRING | URGENTE · ALTA · MEDIA · BAIXA |
| `wind_gust_kmh` | DOUBLE | maior rajada prevista nas 48h (Open-Meteo) |
| `precipitation_mm` | DOUBLE | chuva prevista nas 48h |
| `weather_observed_at` | STRING (ISO 8601) | hora do pico de rajada |
| `recommended_action` | STRING | ação recomendada para Defesa Civil / Enel |
| `audit_actor_hash` | STRING | identificador anonimizado (SHA-256) do lote que gerou o score |

**Frequência:** diária, depois do DAG `arborgrid_coleta` e dos notebooks 01 → 04.
**Forma de leitura:** o pipeline publica o lote em `POST /api/v1/gold/risk-scores` (MERGE idempotente por
`geosampa_id` no MVP). Como a API roda localmente, o lote é exportado para o Volume (`_export/`) e enviado
pelo script `scripts/enviar_ouro_mvp.py`, com o token em variável de ambiente.

In [0]:
SET TIME ZONE 'America/Sao_Paulo';

key,value
spark.sql.session.timeZone,America/Sao_Paulo


In [0]:
CREATE OR REPLACE VIEW workspace.gs.gold_fato_risco_arboreo AS
WITH ref AS (SELECT MAX(sk_data) AS sk FROM workspace.gs.ouro_fato_risco_arboreo)
SELECT
  f.geosampa_id,
  a.distrito,
  CAST(f.composite_risk_score AS DOUBLE)                            AS composite_risk_score,
  f.priority_tier,
  CAST(f.rajada_max_48h AS DOUBLE)                                  AS wind_gust_kmh,
  CAST(f.chuva_48h_mm AS DOUBLE)                                    AS precipitation_mm,
  date_format(f.hora_pico_rajada, "yyyy-MM-dd'T'HH:mm:ssXXX")       AS weather_observed_at,
  f.recommended_action,
  substr(sha2(concat('arborgrid-bigdata-pipeline-', CAST(f.sk_data AS STRING)), 256), 1, 16) AS audit_actor_hash,
  f.sk_data
FROM workspace.gs.ouro_fato_risco_arboreo f
JOIN ref ON f.sk_data = ref.sk
JOIN workspace.gs.ouro_dim_arvore a ON a.geosampa_id = f.geosampa_id;

In [0]:
-- Conferência do contrato (print para o relatório)
SELECT priority_tier, COUNT(*) AS arvores, ROUND(AVG(composite_risk_score), 1) AS score_medio,
       MAX(wind_gust_kmh) AS rajada_max_kmh
FROM workspace.gs.gold_fato_risco_arboreo
GROUP BY priority_tier
ORDER BY score_medio DESC;

priority_tier,arvores,score_medio,rajada_max_kmh
ALTA,106,80.0,35.6
MEDIA,42,56.0,35.6
BAIXA,114,20.7,35.6


## Exporta o lote no formato do `POST /api/v1/gold/risk-scores`

In [0]:
%python
import json, os

df = spark.table("workspace.gs.gold_fato_risco_arboreo")
sk = df.agg({"sk_data": "max"}).first()[0]
registros = [
    {k: v for k, v in r.asDict().items() if k != "sk_data"}
    for r in df.orderBy("geosampa_id").collect()
]
lote = {"batch_id": f"gold-{sk}-arborgrid-bigdata", "records": registros}

destino_dir = "/Volumes/workspace/gs/chegada/_export"
os.makedirs(destino_dir, exist_ok=True)
destino = f"{destino_dir}/gold_risk_scores_{sk}.json"
with open(destino, "w", encoding="utf-8") as f:
    json.dump(lote, f, ensure_ascii=False, indent=2)

print(f"{len(registros)} árvores exportadas para {destino}")
print(json.dumps(registros[0], ensure_ascii=False, indent=2))

262 árvores exportadas para /Volumes/workspace/gs/chegada/_export/gold_risk_scores_20261117.json
{
  "geosampa_id": "1000404",
  "distrito": "PINHEIROS",
  "composite_risk_score": 80.0,
  "priority_tier": "ALTA",
  "wind_gust_kmh": 35.6,
  "precipitation_mm": 4.3,
  "weather_observed_at": "2026-11-17T17:00:00-03:00",
  "recommended_action": "Agendar poda prioritária em até 7 dias",
  "audit_actor_hash": "5d6cc5338c2051da"
}
